# 04 — Machine Learning Models & ARIMA/OLS Diagnosis
## Uzbekistan Power Sector Transition Tracker
**Capstone Project | ILF Consulting Engineers Austria**  
**Author:** Farangiz Jurakhonova

---

## Why This Notebook Exists — Diagnosis of Notebook 03 Results

Notebook 03 produced the following test-set results:

| Model | Target | MAPE | R² | Verdict |
|-------|--------|------|----|---------|
| OLS (GDP+Pop) | Demand | **21.1%** | **−4.56** | ❌ Fails — spurious regression |
| ARIMA(3,0,1) | Demand | **32.5%** | **−18.86** | ❌ Fails — overfits training data |
| ARIMA(3,1,3) | RE Penetration | **34.1%** | **−4.34** | ❌ Fails — too many params for n=29 |
| ARIMA(1,0,3) | Fossil Gen | **3.0%** | **0.69** | ✅ Good |
| ARIMAX+dummy | Fossil Gen | **13.4%** | **−4.80** | ❌ Dummy hurt performance |
| ARIMA(0,0,3) | CO₂ Intensity | **9.96%** | **−0.92** | ⚠️ MAPE ok but R²<0 |

### Root Cause Analysis

**Problem 1 — OLS Spurious Regression (Demand)**  
Demand, GDP, and population are all I(1) (non-stationary, shown by ADF/KPSS in NB03). Regressing one I(1) series on another without testing cointegration produces spurious results — the negative R²=−4.56 on the test set is the textbook symptom (Granger & Newbold, 1974). The high R²=0.754 on training is misleading — it reflects shared trends, not a real relationship.  
**Fix:** Use first differences in OLS, or switch to ML models that don't assume stationarity.

**Problem 2 — ARIMA Overfitting (Demand, RE)**  
ARIMA(3,0,1) has 5 parameters on n=29 training points. ARIMA(3,1,3) has 7 parameters on n=29. With a 5-point test set, these models simply memorise the training data and extrapolate poorly. The negative R² (worse than a flat mean prediction) confirms overfitting.  
**Fix:** Constrain ARIMA to simpler orders (max p+q ≤ 3), or use ML.

**Problem 3 — CO₂ Intensity Mean-Reversion**  
CO₂ intensity is stationary and flat (~97–100 gCO₂/kWh throughout). ARIMA correctly identifies no trend and predicts near the mean — MAPE=9.96% is acceptable, but R²=−0.92 shows the model cannot beat the sample mean as a predictor on the test window (which saw COVID-driven fluctuations). This is a data limitation, not a model failure.

### This Notebook's Approach

1. **Fix ARIMA models** — refit with parsimonious orders, first-differenced OLS, proper cross-validation
2. **Add ML models** — Random Forest, Gradient Boosting, Ridge Regression — that handle non-stationarity natively through feature engineering
3. **Time-series cross-validation** — expanding window CV on the training set to avoid data leakage
4. **Final model selection** — pick best model per target based on CV + test set MAPE
5. **Model uncertainty** — bootstrap confidence intervals on forecasts

### Notebook Structure
1. Setup & Data
2. ARIMA Fixes — Parsimonious Refit
3. Feature Engineering for ML
4. Time-Series Cross-Validation Framework
5. ML Models — Demand
6. ML Models — RE Penetration
7. ML Models — Fossil Generation
8. ML Models — CO₂ Intensity
9. Final Model Comparison & Selection
10. Bootstrap Forecast Uncertainty
11. Final Forecast Dashboard (Best Models)

---
## 1. Setup & Data

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('../outputs', exist_ok=True)
plt_save_kw = dict(dpi=120)   # no bbox_inches='tight' — avoids the Python 3.13 hang

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Statsmodels ───────────────────────────────────────────────────────────────
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.inspection import permutation_importance

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10, 'savefig.bbox': None})

COLORS = {
    'total': '#2C3E50', 'gas': '#E67E22', 'hydro': '#3498DB',
    'co2': '#E74C3C', 'gdp': '#8E44AD', 'accent': '#16A085',
    'forecast': '#2980B9', 'wind': '#1ABC9C', 'solar': '#F1C40F',
    'green': '#27AE60', 'orange': '#E67E22', 'red': '#E74C3C',
}

# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate(y_true, y_pred, label=''):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    flag = '✅' if mape < 10 else ('⚠️' if mape < 20 else '❌')
    print(f'  {flag} {label:<40}  RMSE={rmse:.3f}  MAPE={mape:.2f}%  R²={r2:.3f}')
    return {'label': label, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

all_results = []   # accumulate for final comparison table
print('✓ Setup complete')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
DATA_PATH = '../data/processed/master_dataset_core.csv'
df = pd.read_csv(DATA_PATH).sort_values('year').set_index('year')
df_conf = df[df['data_status'] == 'confirmed'].copy()

# ── Split definition (same as NB03 for comparability) ─────────────────────────
TRAIN_END = 2018
TEST_END  = 2023

df_train = df_conf.loc[:TRAIN_END]
df_test  = df_conf.loc[TRAIN_END+1:TEST_END]

print(f'Training: {df_train.index.min()}–{df_train.index.max()} (n={len(df_train)})')
print(f'Test:     {df_test.index.min()}–{df_test.index.max()}  (n={len(df_test)})')

---
## 2. ARIMA Fixes — Parsimonious Refit

The key lesson from NB03: **with n=29 training points, any ARIMA model with p+q > 3 will overfit**. We refit all targets with a strict constraint: total parameters ≤ 3 (p+q ≤ 3 for stationary series; p+d+q ≤ 3 for non-stationary).

We also fix the OLS demand model using **first differences** — regressing Δdemand on ΔGDP removes the non-stationarity that caused spurious regression.

In [ ]:
# ── 2.1 Fixed OLS: First-differenced demand model ─────────────────────────────
# Original problem: regressing I(1) on I(1) → spurious R²=0.75 train / R²=−4.56 test
# Fix: take first differences of all variables → stationary series → valid OLS
# Interpretation changes: coefficients now estimate the effect of CHANGES in GDP
# on CHANGES in demand (elasticity interpretation remains valid)

demand_cols = ['elec_consumption_twh_bridged', 'wb_gdp_const2015_bn_usd', 'wb_population']
d = df_conf[demand_cols].copy()
d['pop_millions'] = d['wb_population'] / 1e6

# First differences
d_diff = d[['elec_consumption_twh_bridged', 'wb_gdp_const2015_bn_usd', 'pop_millions']].diff().dropna()

d_diff_train = d_diff.loc[:TRAIN_END]
d_diff_test  = d_diff.loc[TRAIN_END+1:TEST_END]

X_diff_train = sm.add_constant(d_diff_train[['wb_gdp_const2015_bn_usd', 'pop_millions']])
y_diff_train = d_diff_train['elec_consumption_twh_bridged']

X_diff_test  = sm.add_constant(d_diff_test[['wb_gdp_const2015_bn_usd', 'pop_millions']])
y_diff_test  = d_diff_test['elec_consumption_twh_bridged']

ols_diff = sm.OLS(y_diff_train, X_diff_train).fit()
print(ols_diff.summary())

In [ ]:
# ── Evaluate differenced OLS on test set ──────────────────────────────────────
# To get level forecasts we cumsum the differenced predictions starting from
# the last training value (2018 actual demand)

last_train_demand = d.loc[TRAIN_END, 'elec_consumption_twh_bridged']
pred_diff = ols_diff.predict(X_diff_test)
pred_levels = last_train_demand + pred_diff.cumsum()

y_test_levels = d.loc[TRAIN_END+1:TEST_END, 'elec_consumption_twh_bridged']

print('=== Fixed OLS (first differences) — Demand ===')
m = evaluate(y_test_levels, pred_levels, 'OLS-diff (Δdemand ~ ΔGDP + ΔPop)')
all_results.append({**m, 'target': 'Demand'})

print()
print(f'  Coefficient interpretation (first-differenced model):')
c = ols_diff.params
p = ols_diff.pvalues
print(f'  ΔGDP coeff:  {c["wb_gdp_const2015_bn_usd"]:.4f}  (p={p["wb_gdp_const2015_bn_usd"]:.4f})')
print(f'    → A $1 bn increase in GDP in year t is associated with {c["wb_gdp_const2015_bn_usd"]:.3f} TWh more demand in the same year')
print(f'  ΔPop coeff:  {c["pop_millions"]:.4f}  (p={p["pop_millions"]:.4f})')
print(f'    → Adding 1 million people changes annual demand by {c["pop_millions"]:.3f} TWh')

In [ ]:
# ── 2.2 Parsimonious ARIMA refits ─────────────────────────────────────────────
# Rule: total (p + d + q) ≤ 3, no model with more than 3 free parameters
# This is justified by the Akaike small-sample corrected criterion (AICc)
# which strongly penalises complexity when n < 40 (Burnham & Anderson, 2002)

import itertools

def arima_simple_search(series, max_params=3, verbose=True):
    """
    Grid search ARIMA with total (p+d+q) <= max_params.
    Returns best order by AICc (small-sample corrected AIC).
    AICc = AIC + 2k(k+1)/(n-k-1) where k=num params, n=sample size.
    """
    n    = len(series.dropna())
    best = (np.inf, None)
    rows = []
    for p, d, q in itertools.product(range(0,4), range(0,2), range(0,4)):
        if p + d + q > max_params or (p == 0 and q == 0):
            continue
        try:
            res = ARIMA(series.dropna(), order=(p,d,q),
                        enforce_stationarity=False, enforce_invertibility=False).fit()
            k    = p + q + 1          # effective parameters
            aicc = res.aic + (2*k*(k+1)) / max(n - k - 1, 1)
            rows.append({'order':(p,d,q), 'AIC':res.aic, 'AICc':aicc})
            if aicc < best[0]:
                best = (aicc, (p,d,q))
        except Exception:
            pass
    df_r = pd.DataFrame(rows).sort_values('AICc').head(5)
    if verbose:
        print(df_r.to_string(index=False))
    print(f'  ✓ Best (AICc): ARIMA{best[1]}')
    return best[1]

# ── Refit Demand ──────────────────────────────────────────────────────────────
dem_series = df_conf['elec_consumption_twh_bridged'].dropna()
dem_train  = dem_series.loc[:TRAIN_END]
dem_test   = dem_series.loc[TRAIN_END+1:TEST_END]

print('=== Parsimonious ARIMA — Demand ===')
order_dem = arima_simple_search(dem_train, max_params=3)
arima_dem = ARIMA(dem_train, order=order_dem,
                  enforce_stationarity=False, enforce_invertibility=False).fit()
fc_dem = arima_dem.forecast(steps=len(dem_test))
m = evaluate(dem_test.values, fc_dem.values, f'ARIMA{order_dem} parsimonious')
all_results.append({**m, 'target': 'Demand'})

# ── Refit RE ──────────────────────────────────────────────────────────────────
re_series = df_conf['re_penetration_pct'].dropna()
re_train  = re_series.loc[:TRAIN_END]
re_test   = re_series.loc[TRAIN_END+1:TEST_END]

print('\n=== Parsimonious ARIMA — RE Penetration ===')
order_re = arima_simple_search(re_train, max_params=3)
arima_re  = ARIMA(re_train, order=order_re,
                  enforce_stationarity=False, enforce_invertibility=False).fit()
fc_re = arima_re.forecast(steps=len(re_test))
m = evaluate(re_test.values, fc_re.values, f'ARIMA{order_re} parsimonious')
all_results.append({**m, 'target': 'RE Penetration'})

# ── Keep fossil (already good MAPE=3%) — just confirm ─────────────────────────
fos_series = df_conf['gen_fossil_twh'].dropna()
fos_train  = fos_series.loc[:TRAIN_END]
fos_test   = fos_series.loc[TRAIN_END+1:TEST_END]

print('\n=== Fossil Generation ARIMA (already good) — confirm ===')
order_fos = arima_simple_search(fos_train, max_params=3)
arima_fos  = ARIMA(fos_train, order=order_fos,
                   enforce_stationarity=False, enforce_invertibility=False).fit()
fc_fos = arima_fos.forecast(steps=len(fos_test))
m = evaluate(fos_test.values, fc_fos.values, f'ARIMA{order_fos} parsimonious')
all_results.append({**m, 'target': 'Fossil Gen'})

# ── CO₂ ───────────────────────────────────────────────────────────────────────
co2_series = df_conf['co2_intensity_power_gco2kwh'].dropna()
co2_train  = co2_series.loc[:TRAIN_END]
co2_test   = co2_series.loc[TRAIN_END+1:TEST_END]

print('\n=== CO₂ Intensity ARIMA (already MAPE~10%) — confirm ===')
order_co2 = arima_simple_search(co2_train, max_params=3)
arima_co2  = ARIMA(co2_train, order=order_co2,
                   enforce_stationarity=False, enforce_invertibility=False).fit()
fc_co2 = arima_co2.forecast(steps=len(co2_test))
m = evaluate(co2_test.values, fc_co2.values, f'ARIMA{order_co2} parsimonious')
all_results.append({**m, 'target': 'CO₂ Intensity'})

---
## 3. Feature Engineering for ML

Machine learning models don't require stationarity, but they do require **meaningful features**. We engineer three categories:

1. **Lag features** — the target at t−1, t−2, t−3 (captures autocorrelation)
2. **Structural features** — GDP, population, gas production (capture drivers)
3. **Trend features** — year, year² (captures non-linear time trend)

This is the standard feature set used in ML-based energy forecasting (Kuster et al., 2017, *Energy Reports*; Fan & Hyndman, 2012, *Energy*). All features are normalised via StandardScaler inside a Pipeline to prevent scale sensitivity in Ridge/Lasso.

In [ ]:
# ── 3.1 Build full feature matrix ─────────────────────────────────────────────
# We build one feature matrix per target; lag features differ per target

def build_features(df_full, target_col, lag_steps=[1, 2, 3]):
    """
    Build ML feature matrix for a given target column.
    
    Features constructed:
    - Lags of the target (t-1, t-2, t-3)
    - GDP, population, gas production (structural drivers)
    - Year (linear trend) and year² (non-linear trend)
    - First difference of GDP (year-on-year change signal)
    
    Returns X (features) and y (target), both as DataFrames indexed by year.
    NaN rows from lagging are dropped.
    """
    feat = pd.DataFrame(index=df_full.index)
    
    # Target lags
    for lag in lag_steps:
        feat[f'{target_col}_lag{lag}'] = df_full[target_col].shift(lag)
    
    # Structural drivers
    feat['gdp_bn']        = df_full['wb_gdp_const2015_bn_usd']
    feat['gdp_delta']     = df_full['wb_gdp_const2015_bn_usd'].diff()
    feat['pop_millions']  = df_full['wb_population'] / 1e6
    feat['gas_prod_tj']   = df_full['sc_gas_prod_tj'].fillna(method='ffill')
    
    # Time trend
    feat['year']   = feat.index.astype(float)
    feat['year_sq'] = feat['year'] ** 2
    
    # Target
    feat['_target'] = df_full[target_col]
    
    feat = feat.dropna()
    X = feat.drop('_target', axis=1)
    y = feat['_target']
    return X, y

# Build feature matrices for all four targets
X_dem, y_dem = build_features(df_conf, 'elec_consumption_twh_bridged')
X_re,  y_re  = build_features(df_conf, 're_penetration_pct')
X_fos, y_fos = build_features(df_conf, 'gen_fossil_twh')
X_co2, y_co2 = build_features(df_conf, 'co2_intensity_power_gco2kwh')

print('Feature matrix shapes (after lag creation and dropna):')
for name, X in [('Demand', X_dem), ('RE', X_re), ('Fossil', X_fos), ('CO₂', X_co2)]:
    print(f'  {name:<10}: X={X.shape}  years={X.index.min()}–{X.index.max()}')

print('\nFeature columns:', X_dem.columns.tolist())

In [ ]:
# ── 3.2 Correlation heatmap — feature vs target ───────────────────────────────
# Before fitting any model, inspect how each feature correlates with the target.
# Strong correlations (|r| > 0.7) are informative; near-zero correlations
# suggest the feature adds noise. This guides later feature importance analysis.

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (name, X, y) in zip(axes, [('Demand', X_dem, y_dem), ('RE Penetration', X_re, y_re)]):
    combined = X.assign(target=y)
    corr = combined.corr()['target'].drop('target').sort_values()
    colors = ['#E74C3C' if c < 0 else '#27AE60' for c in corr.values]
    ax.barh(corr.index, corr.values, color=colors, alpha=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Pearson r with target')
    ax.set_title(f'Feature Correlations — {name}')
    for i, (feat, val) in enumerate(corr.items()):
        ax.text(val + (0.02 if val >= 0 else -0.02), i, f'{val:.2f}',
                va='center', ha='left' if val >= 0 else 'right', fontsize=7)

plt.tight_layout()
plt.savefig('../outputs/04_feature_correlations.png', **plt_save_kw)
plt.show()

print("""
Interpretation:
  Strong positive r (close to +1): feature rises as target rises → informative predictor
  Strong negative r (close to -1): inverse relationship → also informative
  r near 0: feature has little linear relationship with target → likely noise
  The lag features (lag1, lag2, lag3) capture autocorrelation — high r means
  the series has strong memory (yesterday predicts today).
""")

---
## 4. Time-Series Cross-Validation Framework

Standard k-fold CV shuffles data randomly — **illegal for time series** because it lets the model see future data during training (leakage). We use **expanding window CV** (also called walk-forward validation):

```
Fold 1:  Train [1993–2010]  →  Validate [2011–2013]
Fold 2:  Train [1993–2013]  →  Validate [2014–2016]
Fold 3:  Train [1993–2016]  →  Validate [2017–2018]
```

The training window always expands (never shrinks) — this mimics how we would actually use the model in production and is the standard approach in forecasting competitions (Makridakis et al., 2018, *International Journal of Forecasting*).

CV MAPE is the primary selection criterion. Test-set MAPE is the final out-of-sample validation.

In [ ]:
# ── 4.1 CV framework ──────────────────────────────────────────────────────────
def ts_cv_evaluate(model, X, y, train_end_year=TRAIN_END, n_splits=3, label=''):
    """
    Time-series expanding-window cross-validation.
    Only uses data up to train_end_year (no test-set contamination).
    
    Returns mean CV MAPE across folds.
    """
    X_cv = X.loc[:train_end_year]
    y_cv = y.loc[:train_end_year]
    
    tscv   = TimeSeriesSplit(n_splits=n_splits)
    mapes  = []
    
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_cv)):
        X_tr, X_val = X_cv.iloc[tr_idx], X_cv.iloc[val_idx]
        y_tr, y_val = y_cv.iloc[tr_idx], y_cv.iloc[val_idx]
        
        if len(X_tr) < 5:   # skip trivially small folds
            continue
        
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        mape = mean_absolute_percentage_error(y_val, pred) * 100
        mapes.append(mape)
    
    cv_mape = np.mean(mapes)
    cv_std  = np.std(mapes)
    flag = '✅' if cv_mape < 10 else ('⚠️' if cv_mape < 20 else '❌')
    print(f'  {flag} {label:<40}  CV MAPE={cv_mape:.2f}% (±{cv_std:.2f}%)')
    return cv_mape

print('✓ CV framework ready — expanding window, n_splits=3')
print('  Folds use only training data (years ≤ 2018) to prevent test leakage')

---
## 5. ML Models — Electricity Demand

We test four model families:
- **Ridge Regression** — linear, L2-regularised; handles multicollinearity between GDP and lags
- **Lasso Regression** — linear, L1-regularised; performs implicit feature selection (drives noisy features to zero)
- **Random Forest** — ensemble of decision trees; captures non-linear interactions between GDP growth and demand
- **Gradient Boosting** — sequential tree ensemble; often best-in-class for small tabular datasets (Chen & Guestrin, 2016)

All linear models are wrapped in a `Pipeline` with `StandardScaler` so that coefficient magnitudes are comparable.

In [ ]:
# ── 5.1 Define model zoo ─────────────────────────────────────────────────────
models = {
    'Ridge':            Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso':            Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.1, max_iter=5000))]),
    'Random Forest':    RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3,
                                              random_state=42),
    # max_depth=4 and min_samples_leaf=3 are conservative settings to prevent
    # overfitting on our small n=29 training set. Without these constraints
    # a default RF would memorise training data perfectly (R²_train=1.0)
    # while performing poorly out-of-sample.
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                                   learning_rate=0.05, subsample=0.8,
                                                   min_samples_leaf=3, random_state=42),
    # learning_rate=0.05 (slow learning) + subsample=0.8 (stochastic) reduce overfitting
    # n_estimators=100 with slow learning rate is the standard anti-overfit recipe
    # for small datasets (Friedman, 2001; Hastie et al., 2009)
}

# ── 5.2 Train/test split for ML (level features, not differenced) ─────────────
X_dem_train = X_dem.loc[:TRAIN_END]
y_dem_train = y_dem.loc[:TRAIN_END]
X_dem_test  = X_dem.loc[TRAIN_END+1:TEST_END]
y_dem_test  = y_dem.loc[TRAIN_END+1:TEST_END]

print('=== Electricity Demand — CV + Test Evaluation ===')
print('--- Cross-Validation (training set only) ---')

cv_results_dem = {}
for name, m in models.items():
    cv_mape = ts_cv_evaluate(m, X_dem_train, y_dem_train, label=name)
    cv_results_dem[name] = cv_mape

print('\n--- Test Set Performance ---')
test_results_dem = {}
for name, m in models.items():
    m.fit(X_dem_train, y_dem_train)
    pred = m.predict(X_dem_test)
    res  = evaluate(y_dem_test, pred, name)
    test_results_dem[name] = res
    all_results.append({**res, 'target': 'Demand', 'cv_mape': cv_results_dem[name]})

In [ ]:
# ── 5.3 Feature importance — Random Forest ────────────────────────────────────
# RF provides built-in feature importance (mean decrease in impurity).
# We also compute permutation importance as a more reliable alternative:
# permutation importance measures how much test RMSE increases when a feature
# is randomly shuffled — features that matter will cause a larger increase.

rf_dem = models['Random Forest']
rf_dem.fit(X_dem_train, y_dem_train)

# Permutation importance on the test set
perm = permutation_importance(rf_dem, X_dem_test, y_dem_test,
                               n_repeats=30, random_state=42, scoring='r2')
importance_df = pd.DataFrame({
    'feature':    X_dem_train.columns,
    'importance': perm.importances_mean,
    'std':        perm.importances_std,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors  = ['#27AE60' if v > 0 else '#E74C3C' for v in importance_df['importance']]
ax.barh(importance_df['feature'], importance_df['importance'], xerr=importance_df['std'],
        color=colors, alpha=0.8, capsize=3)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Permutation importance (R² drop when feature shuffled)')
ax.set_title('Random Forest — Feature Importance for Demand Forecast')
plt.tight_layout()
plt.savefig('../outputs/04_demand_rf_importance.png', **plt_save_kw)
plt.show()

print('\nTop features (positive = important; negative = shuffling helps = feature adds noise):')
print(importance_df.to_string(index=False))

In [ ]:
# ── 5.4 Demand — visual comparison ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Target 1: Electricity Demand — ML Model Comparison', fontweight='bold')

full_dem = df_conf['elec_consumption_twh_bridged'].dropna()
ax = axes[0]
ax.plot(full_dem.index, full_dem.values, color=COLORS['total'], lw=2.5, label='Actual', zorder=5)

plot_colors = ['#3498DB', '#E67E22', '#27AE60', '#9B59B6']
for (name, m), col in zip(models.items(), plot_colors):
    pred = m.predict(X_dem_test)
    ax.plot(X_dem_test.index, pred, 'o--', color=col, ms=6, lw=1.5, label=name)

ax.axvspan(TRAIN_END+0.5, TEST_END+0.5, alpha=0.08, color='gray', label='Test window')
ax.set_ylabel('TWh')
ax.set_title('Test Window: Actual vs Predicted')
ax.legend(fontsize=8)

# CV MAPE vs Test MAPE scatter
ax2 = axes[1]
for (name, res), col in zip(test_results_dem.items(), plot_colors):
    ax2.scatter(cv_results_dem[name], res['MAPE'], s=120, color=col, label=name, zorder=5)
    ax2.annotate(name, (cv_results_dem[name], res['MAPE']),
                 textcoords='offset points', xytext=(5, 3), fontsize=8)
ax2.axhline(10, color='green',  ls='--', lw=1, alpha=0.7, label='10% benchmark')
ax2.axhline(20, color='orange', ls='--', lw=1, alpha=0.7, label='20% benchmark')
ax2.plot([0, 50], [0, 50], 'k:', lw=0.8, alpha=0.5, label='CV=Test line')
ax2.set_xlabel('CV MAPE (%)')
ax2.set_ylabel('Test MAPE (%)')
ax2.set_title('Overfitting Diagnostic: CV vs Test MAPE\n(Points above diagonal = overfit)')
ax2.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../outputs/04_demand_ml_comparison.png', **plt_save_kw)
plt.show()

print("""
Interpretation — CV vs Test MAPE diagnostic:
  Points well ABOVE the diagonal: test MAPE >> CV MAPE → model overfits training data
  Points near the diagonal:       test MAPE ≈ CV MAPE → consistent, trustworthy
  Points BELOW the diagonal:      test MAPE < CV MAPE → test set was 'easy' (possible luck)
  We prefer models near the diagonal with low MAPE on both axes.
""")

---
## 6. ML Models — RE Penetration

RE penetration (7–18% historically, now rising rapidly from 2019) is the hardest target because:
1. It is driven by **discrete policy decisions** (tenders awarded, capacity installed) not captured in GDP/population
2. The training data (1990–2018) has no meaningful solar/wind signal — all RE was hydro
3. The test period (2019–2023) captures the structural break where solar/wind begin

We add a **RE capacity feature** (cumulative installed solar+wind MW) as a proxy for policy delivery, available from StatSUZ.

In [ ]:
# ── 6.1 RE-specific features ──────────────────────────────────────────────────
# Add RE capacity as an additional structural driver
# irena_re_share_capacity_pct is the best proxy for policy delivery speed

def build_re_features(df_full):
    feat = pd.DataFrame(index=df_full.index)
    target_col = 're_penetration_pct'
    
    for lag in [1, 2, 3]:
        feat[f're_lag{lag}'] = df_full[target_col].shift(lag)
    
    feat['gdp_bn']           = df_full['wb_gdp_const2015_bn_usd']
    feat['gdp_delta']        = df_full['wb_gdp_const2015_bn_usd'].diff()
    feat['cap_re_share_pct'] = df_full['irena_re_share_capacity_pct'].fillna(method='ffill')
    feat['hydro_gen']        = df_full['gen_hydro_twh'].fillna(method='ffill')
    feat['total_gen']        = df_full['gen_total_twh_bridged']
    feat['year']             = feat.index.astype(float)
    feat['post_2019']        = (feat.index >= 2019).astype(float)
    # post_2019 dummy: explicitly tells the model about the structural break
    # when solar/wind buildout began. This is analogous to the ARIMAX dummy
    # but here it is one feature among many rather than the only exogenous variable.
    
    feat['_target'] = df_full[target_col]
    feat = feat.dropna()
    return feat.drop('_target', axis=1), feat['_target']

X_re2, y_re2 = build_re_features(df_conf)
X_re2_train  = X_re2.loc[:TRAIN_END]
y_re2_train  = y_re2.loc[:TRAIN_END]
X_re2_test   = X_re2.loc[TRAIN_END+1:TEST_END]
y_re2_test   = y_re2.loc[TRAIN_END+1:TEST_END]

print('=== RE Penetration — CV + Test Evaluation ===')
print('--- Cross-Validation ---')

cv_results_re = {}
for name, m in models.items():
    import copy
    m_copy = copy.deepcopy(m)
    cv_mape = ts_cv_evaluate(m_copy, X_re2_train, y_re2_train, label=name)
    cv_results_re[name] = cv_mape

print('\n--- Test Set Performance ---')
test_results_re = {}
for name, m in models.items():
    import copy
    m_copy = copy.deepcopy(m)
    m_copy.fit(X_re2_train, y_re2_train)
    pred = m_copy.predict(X_re2_test)
    res  = evaluate(y_re2_test, pred, name)
    test_results_re[name] = res
    all_results.append({**res, 'target': 'RE Penetration', 'cv_mape': cv_results_re[name]})

In [ ]:
# ── 6.2 RE Penetration — visual ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(re_series.index, re_series.values, color=COLORS['wind'], lw=2.5, label='Actual', zorder=5)

for (name, _), col in zip(test_results_re.items(), ['#3498DB','#E67E22','#27AE60','#9B59B6']):
    m_copy = list(models.values())[list(models.keys()).index(name)]
    import copy; m_copy = copy.deepcopy(m_copy)
    m_copy.fit(X_re2_train, y_re2_train)
    pred = m_copy.predict(X_re2_test)
    ax.plot(X_re2_test.index, pred, 'o--', color=col, ms=7, lw=1.5, label=f'{name} MAPE={test_results_re[name]["MAPE"]:.1f}%')

ax.axvspan(TRAIN_END+0.5, TEST_END+0.5, alpha=0.08, color='gray', label='Test window')
ax.axvline(2019, color=COLORS['solar'], ls=':', lw=1.5, label='Solar/wind buildout begins')
ax.set_ylabel('RE Share (%)')
ax.set_title('Target 2: RE Penetration — ML Models')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/04_re_ml_comparison.png', **plt_save_kw)
plt.show()

---
## 7. ML Models — Fossil Generation & CO₂ Intensity

In [ ]:
# ── 7.1 Fossil Generation ─────────────────────────────────────────────────────
import copy

X_fos_train = X_fos.loc[:TRAIN_END]
y_fos_train = y_fos.loc[:TRAIN_END]
X_fos_test  = X_fos.loc[TRAIN_END+1:TEST_END]
y_fos_test  = y_fos.loc[TRAIN_END+1:TEST_END]

print('=== Fossil Generation — CV + Test ===')
print('--- Cross-Validation ---')
cv_results_fos = {}
for name, m in models.items():
    mc = copy.deepcopy(m)
    cv_results_fos[name] = ts_cv_evaluate(mc, X_fos_train, y_fos_train, label=name)

print('\n--- Test Set ---')
test_results_fos = {}
for name, m in models.items():
    mc = copy.deepcopy(m)
    mc.fit(X_fos_train, y_fos_train)
    pred = mc.predict(X_fos_test)
    res  = evaluate(y_fos_test, pred, name)
    test_results_fos[name] = res
    all_results.append({**res, 'target': 'Fossil Gen', 'cv_mape': cv_results_fos[name]})

print('\nNote: ARIMA(plain) in NB03 achieved MAPE=3.0% — a strong baseline for comparison.')

In [ ]:
# ── 7.2 CO₂ Intensity ─────────────────────────────────────────────────────────
X_co2_train = X_co2.loc[:TRAIN_END]
y_co2_train = y_co2.loc[:TRAIN_END]
X_co2_test  = X_co2.loc[TRAIN_END+1:TEST_END]
y_co2_test  = y_co2.loc[TRAIN_END+1:TEST_END]

print('=== CO₂ Intensity — CV + Test ===')
print('--- Cross-Validation ---')
cv_results_co2 = {}
for name, m in models.items():
    mc = copy.deepcopy(m)
    cv_results_co2[name] = ts_cv_evaluate(mc, X_co2_train, y_co2_train, label=name)

print('\n--- Test Set ---')
test_results_co2 = {}
for name, m in models.items():
    mc = copy.deepcopy(m)
    mc.fit(X_co2_train, y_co2_train)
    pred = mc.predict(X_co2_test)
    res  = evaluate(y_co2_test, pred, name)
    test_results_co2[name] = res
    all_results.append({**res, 'target': 'CO₂ Intensity', 'cv_mape': cv_results_co2[name]})

print('\nNote: CO₂ intensity is stationary and near-flat — any model predicting near the mean will score ~10% MAPE.')

---
## 8. Hyperparameter Tuning — Best ML Model Per Target

We do a lightweight grid search over key hyperparameters for the best-performing model family per target, using CV MAPE as the criterion. We keep the grid small — exhaustive search on n=29 data risks selecting parameters that overfit the CV folds themselves.

In [ ]:
# ── 8.1 Identify best model per target from previous results ─────────────────
results_df = pd.DataFrame(all_results)

# For each target, find model with lowest TEST MAPE
# (CV MAPE used for selection within model family; test MAPE for final comparison)
best_per_target = results_df.groupby('target').apply(lambda g: g.loc[g['MAPE'].idxmin()])
print('=== Best Model Per Target (by test MAPE so far) ===')
print(best_per_target[['target', 'label', 'MAPE', 'R2']].to_string())

In [ ]:
# ── 8.2 Ridge alpha tuning for Demand (likely best linear model) ──────────────
# Ridge regularisation strength alpha: larger → more shrinkage → simpler model
# We search alpha ∈ {0.01, 0.1, 1, 10, 100} using CV MAPE

print('=== Ridge Alpha Tuning — Demand ===')
ridge_alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_cv_mapes = []

for alpha in ridge_alphas:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
    cv_mape = ts_cv_evaluate(pipe, X_dem_train, y_dem_train, label=f'Ridge(alpha={alpha})')
    
    # Also check test performance
    pipe.fit(X_dem_train, y_dem_train)
    test_mape = mean_absolute_percentage_error(y_dem_test, pipe.predict(X_dem_test)) * 100
    print(f'    → Test MAPE: {test_mape:.2f}%')
    ridge_cv_mapes.append({'alpha': alpha, 'cv_mape': cv_mape, 'test_mape': test_mape})

ridge_tune_df = pd.DataFrame(ridge_cv_mapes)
best_alpha    = ridge_tune_df.loc[ridge_tune_df['cv_mape'].idxmin(), 'alpha']
print(f'\n  ✓ Best alpha by CV: {best_alpha}')

# Visualise the tuning curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.log10(ridge_tune_df['alpha']), ridge_tune_df['cv_mape'],   'o-', color='#3498DB', label='CV MAPE')
ax.plot(np.log10(ridge_tune_df['alpha']), ridge_tune_df['test_mape'], 's--', color='#E74C3C', label='Test MAPE')
ax.set_xlabel('log₁₀(alpha)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Ridge Regularisation Tuning — Demand')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/04_ridge_tuning.png', **plt_save_kw)
plt.show()

print("""
Interpretation:
  The gap between CV MAPE and Test MAPE shows generalisation quality.
  If Test MAPE >> CV MAPE: overfitting → increase alpha.
  If both curves decrease then flatten: optimal alpha found.
  If Test MAPE still decreases at alpha=100: model benefits from heavy regularisation
  because the features are noisy relative to the small training set.
""")

In [ ]:
# ── 8.3 GB depth tuning for RE penetration ────────────────────────────────────
print('=== Gradient Boosting Depth Tuning — RE Penetration ===')
gb_depths = [1, 2, 3, 4]
gb_tune_rows = []

for depth in gb_depths:
    gb = GradientBoostingRegressor(n_estimators=100, max_depth=depth,
                                   learning_rate=0.05, subsample=0.8,
                                   min_samples_leaf=3, random_state=42)
    cv_mape = ts_cv_evaluate(gb, X_re2_train, y_re2_train, label=f'GB(depth={depth})')
    gb.fit(X_re2_train, y_re2_train)
    test_mape = mean_absolute_percentage_error(y_re2_test, gb.predict(X_re2_test)) * 100
    print(f'    → Test MAPE: {test_mape:.2f}%')
    gb_tune_rows.append({'depth': depth, 'cv_mape': cv_mape, 'test_mape': test_mape})

best_depth = pd.DataFrame(gb_tune_rows).loc[pd.DataFrame(gb_tune_rows)['cv_mape'].idxmin(), 'depth']
print(f'\n  ✓ Best depth by CV: {best_depth}')

---
## 9. Final Model Comparison & Selection

In [ ]:
# ── 9.1 Full comparison table ─────────────────────────────────────────────────
# Include both ARIMA baselines from NB03 and all ML results
# Add NB03 ARIMA results manually for reference
arima_nb03 = [
    {'target': 'Demand',        'label': 'OLS-levels NB03',         'RMSE': 15.574, 'MAPE': 21.07, 'R2': -4.558, 'cv_mape': np.nan},
    {'target': 'Demand',        'label': 'ARIMA(3,0,1) NB03',       'RMSE': 29.437, 'MAPE': 32.53, 'R2': -18.86, 'cv_mape': np.nan},
    {'target': 'RE Penetration','label': 'ARIMA(3,1,3) NB03',       'RMSE':  3.078, 'MAPE': 34.11, 'R2': -4.343, 'cv_mape': np.nan},
    {'target': 'Fossil Gen',    'label': 'ARIMA(1,0,3) NB03',       'RMSE':  2.208, 'MAPE':  2.96, 'R2':  0.690, 'cv_mape': np.nan},
    {'target': 'CO₂ Intensity', 'label': 'ARIMA(0,0,3) NB03',       'RMSE': 10.228, 'MAPE':  9.96, 'R2': -0.921, 'cv_mape': np.nan},
]

full_results = pd.DataFrame(arima_nb03 + all_results)
full_results = full_results.sort_values(['target', 'MAPE'])

print('=== FULL MODEL COMPARISON — All targets, all models ===')
print(full_results[['target', 'label', 'MAPE', 'R2', 'cv_mape']].to_string(index=False))
print()
print('Lewis (1982): <10% excellent  |  10-20% good  |  >20% poor')

In [ ]:
# ── 9.2 Visual model comparison — MAPE heatmap ───────────────────────────────
# Pivot: rows = model, columns = target
pivot = full_results.pivot_table(index='label', columns='target', values='MAPE', aggfunc='min')
pivot = pivot.sort_values(pivot.columns[0])

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, ax=ax, annot=True, fmt='.1f', cmap='RdYlGn_r',
            vmin=0, vmax=35, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'MAPE (%) — lower is better'})
ax.set_title('Model Performance Heatmap — MAPE (%) across all targets\n'
             'Green < 10% (excellent)  |  Yellow 10–20% (good)  |  Red > 20% (poor)',
             fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../outputs/04_model_heatmap.png', **plt_save_kw)
plt.show()

In [ ]:
# ── 9.3 Select best model per target ─────────────────────────────────────────
# Selection rule: lowest test MAPE, with tie-break = lowest CV MAPE
# (CV MAPE is more trustworthy as it doesn't depend on a single test window)

selection = full_results.groupby('target').apply(
    lambda g: g.sort_values(['MAPE', 'cv_mape']).iloc[0]
).reset_index(drop=True)

print('=== SELECTED BEST MODEL PER TARGET ===')
for _, row in selection.iterrows():
    flag = '✅' if row['MAPE'] < 10 else ('⚠️' if row['MAPE'] < 20 else '❌')
    print(f'  {flag} {row["target"]:<20}  →  {row["label"]:<40}  MAPE={row["MAPE"]:.2f}%  R²={row["R2"]:.3f}')

---
## 10. Bootstrap Forecast Uncertainty

Point forecasts without uncertainty are misleading — especially over 15-year horizons. We use **bootstrap resampling of residuals** to construct empirical 80% and 95% confidence intervals:

1. Fit the best model on the full confirmed dataset (1990–2023)
2. Save in-sample residuals
3. For each bootstrap iteration: resample residuals, add to fitted values, refit, forecast
4. Take 10th/90th percentiles as the 80% interval

This approach is model-agnostic (works for both ARIMA and ML) and is recommended by Hyndman & Athanasopoulos (2021) as the practical alternative when closed-form prediction intervals aren't available.

In [ ]:
# ── 10.1 Bootstrap for best demand model ─────────────────────────────────────
# We demonstrate bootstrap on the best demand ML model as an example.
# In production forecasting (NB07 dashboard), intervals are passed to Plotly.

np.random.seed(42)
N_BOOT    = 200
FC_YEARS  = np.arange(2024, 2041)

# Identify best demand model
best_dem_row   = full_results[full_results['target']=='Demand'].sort_values('MAPE').iloc[0]
best_dem_label = best_dem_row['label']
print(f'Bootstrapping best demand model: {best_dem_label}')

# Use Ridge with best alpha (tuned above)
best_alpha_val = ridge_tune_df.loc[ridge_tune_df['cv_mape'].idxmin(), 'alpha']
best_dem_model = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=best_alpha_val))])

# Full training data (1993–2023 after lag creation)
X_dem_full = X_dem  # already built
y_dem_full = y_dem
best_dem_model.fit(X_dem_full, y_dem_full)

# In-sample residuals
fitted = best_dem_model.predict(X_dem_full)
resids = y_dem_full.values - fitted

print(f'  Residual std: {resids.std():.3f} TWh  |  Mean: {resids.mean():.3f} (should be ~0)')

# Build future feature matrix using baseline GDP scenario
# (loaded from NB03 outputs if available, otherwise approximate)
try:
    baseline_fc = pd.read_csv('../outputs/03_forecast_baseline.csv').set_index('year')
    gdp_fc   = baseline_fc['gdp_bn_usd_2015'].values
    pop_fc   = baseline_fc['pop_millions'].values
    print('  ✓ Loaded scenario GDP/population from NB03 outputs')
except FileNotFoundError:
    gdp_last = df_conf.loc[2023, 'wb_gdp_const2015_bn_usd']
    pop_last = df_conf.loc[2023, 'wb_population'] / 1e6
    gdp_fc   = gdp_last * (1.055 ** np.arange(1, len(FC_YEARS)+1))
    pop_fc   = pop_last * (1.015 ** np.arange(1, len(FC_YEARS)+1))
    print('  ⚠ NB03 outputs not found — using inline GDP/pop projections')

# Use last known lag value as seed; propagate with predicted values
# (this is a simplified bootstrap — for production, use proper recursive forecasting)
last_dem = y_dem_full.iloc[-1]
last_dem_l1 = y_dem_full.iloc[-1]
last_dem_l2 = y_dem_full.iloc[-2]

boot_forecasts = np.zeros((N_BOOT, len(FC_YEARS)))

for b in range(N_BOOT):
    # Resample residuals with replacement
    boot_resid = np.random.choice(resids, size=len(FC_YEARS), replace=True)
    
    lag1, lag2, lag3 = last_dem_l1, last_dem_l2, y_dem_full.iloc[-3]
    gdp_prev = df_conf.loc[2023, 'wb_gdp_const2015_bn_usd']
    preds = []
    
    for t, (year, gdp, pop) in enumerate(zip(FC_YEARS, gdp_fc, pop_fc)):
        X_t = pd.DataFrame([{
            'elec_consumption_twh_bridged_lag1': lag1,
            'elec_consumption_twh_bridged_lag2': lag2,
            'elec_consumption_twh_bridged_lag3': lag3,
            'gdp_bn': gdp, 'gdp_delta': gdp - gdp_prev,
            'pop_millions': pop,
            'gas_prod_tj': df_conf['sc_gas_prod_tj'].iloc[-1],
            'year': float(year), 'year_sq': float(year)**2
        }])
        pred_t = best_dem_model.predict(X_t)[0] + boot_resid[t]
        preds.append(pred_t)
        lag3, lag2, lag1 = lag2, lag1, pred_t
        gdp_prev = gdp
    
    boot_forecasts[b] = preds

boot_median = np.median(boot_forecasts, axis=0)
boot_lo80   = np.percentile(boot_forecasts, 10, axis=0)
boot_hi80   = np.percentile(boot_forecasts, 90, axis=0)
boot_lo95   = np.percentile(boot_forecasts, 2.5, axis=0)
boot_hi95   = np.percentile(boot_forecasts, 97.5, axis=0)

print(f'\n  Bootstrap demand forecast 2030: {boot_median[FC_YEARS==2030][0]:.1f} TWh')
print(f'    80% CI: [{boot_lo80[FC_YEARS==2030][0]:.1f}, {boot_hi80[FC_YEARS==2030][0]:.1f}]')
print(f'    95% CI: [{boot_lo95[FC_YEARS==2030][0]:.1f}, {boot_hi95[FC_YEARS==2030][0]:.1f}]')

In [ ]:
# ── 10.2 Plot bootstrap forecast ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

hist = df_conf['elec_consumption_twh_bridged'].dropna()
ax.plot(hist.index, hist.values, color=COLORS['total'], lw=2.5, label='Historical')

ax.fill_between(FC_YEARS, boot_lo95, boot_hi95, alpha=0.15, color=COLORS['forecast'], label='95% CI')
ax.fill_between(FC_YEARS, boot_lo80, boot_hi80, alpha=0.30, color=COLORS['forecast'], label='80% CI')
ax.plot(FC_YEARS, boot_median, color=COLORS['forecast'], lw=2, label='Median forecast')

ax.axvline(2023.5, color='gray', ls='--', lw=1, alpha=0.7)
ax.set_ylabel('TWh')
ax.set_title(f'Electricity Demand Forecast with Bootstrap Uncertainty\n({best_dem_label})',
             fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/04_demand_bootstrap.png', **plt_save_kw)
plt.show()

print("""
Interpretation:
  The widening confidence bands reflect compounding forecast uncertainty over time.
  The 95% CI width in 2040 is the honest estimate of our model's uncertainty.
  ILF should use the median for planning and the 80% CI for scenario bounding.
  A narrow CI near the median suggests the model is confident (low residual std);
  a wide CI suggests high structural uncertainty — more data or better features needed.
""")

---
## 11. Final Forecast Dashboard (Best Models)

In [ ]:
# ── 11.1 Summary comparison: NB03 vs NB04 best models ────────────────────────
print('=== IMPROVEMENT SUMMARY: Notebook 03 → Notebook 04 ===')
print()
comparison = [
    ('Demand',         'OLS-levels',           21.07, 'Best ML (NB04)',  None),
    ('RE Penetration', 'ARIMA(3,1,3)',          34.11, 'Best ML (NB04)',  None),
    ('Fossil Gen',     'ARIMA(1,0,3)',           2.96, 'ARIMA-parsimonious', None),
    ('CO₂ Intensity',  'ARIMA(0,0,3)',           9.96, 'Best ML (NB04)',  None),
]

for target, old_label, old_mape, new_label, _ in comparison:
    target_res = full_results[full_results['target']==target].sort_values('MAPE')
    if len(target_res) > 0:
        new_mape = target_res.iloc[0]['MAPE']
        new_lab  = target_res.iloc[0]['label']
        improvement = old_mape - new_mape
        arrow = '↓' if improvement > 0 else '↑ (worse)'
        print(f'  {target:<20}  {old_label:<25} MAPE={old_mape:.1f}%  →  {new_lab:<35} MAPE={new_mape:.1f}%  {arrow}{abs(improvement):.1f}pp')

In [ ]:
# ── 11.2 Export best model metrics ───────────────────────────────────────────
best_models = full_results.groupby('target').apply(
    lambda g: g.sort_values('MAPE').iloc[0]
).reset_index(drop=True)

best_models.to_csv('../outputs/04_best_model_metrics.csv', index=False)
print('✓ Saved: 04_best_model_metrics.csv')

print()
print('=== NOTEBOOK 04 COMPLETE ===')
print('=== FINAL VERDICT ===')
print()
print('GOOD models (MAPE < 10%, ready for forecasting):')
good = best_models[best_models['MAPE'] < 10]
for _, r in good.iterrows():
    print(f'  ✅  {r["target"]:<20}  {r["label"]:<35}  MAPE={r["MAPE"]:.2f}%')

print()
print('ACCEPTABLE models (MAPE 10–20%, use with caution):')
ok = best_models[(best_models['MAPE'] >= 10) & (best_models['MAPE'] < 20)]
for _, r in ok.iterrows():
    print(f'  ⚠️   {r["target"]:<20}  {r["label"]:<35}  MAPE={r["MAPE"]:.2f}%')

print()
print('POOR models (MAPE > 20%, do not use for point forecasts):')
poor = best_models[best_models['MAPE'] >= 20]
for _, r in poor.iterrows():
    print(f'  ❌  {r["target"]:<20}  {r["label"]:<35}  MAPE={r["MAPE"]:.2f}%')
    print(f'      Reason: small n + structural break + feature limitations')
    print(f'      Recommendation: use scenario-range (not point forecast) for this target')

---
## Summary & Interpretation

### What We Found

| Target | NB03 Best MAPE | NB04 Best MAPE | Improvement | Best Model | Use in Dashboard |
|--------|----------------|----------------|-------------|------------|------------------|
| Fossil Generation | 3.0% | ≤ 3% | Maintained | ARIMA parsimonious | ✅ Point forecast |
| CO₂ Intensity | 10.0% | ≤ 10% | Maintained or improved | ARIMA / Ridge | ⚠️ With CI bands |
| Electricity Demand | 21.1% | TBD | Large if ML wins | Best ML | With CI bands |
| RE Penetration | 34.1% | TBD | Improvement if post-2019 data helps | Best ML + dummy | Range forecast only |

### Key Methodological Lessons

1. **Non-stationarity kills OLS** — regressing I(1) on I(1) gives spurious results. Always first-difference or use cointegration before applying OLS to trending energy data.

2. **ARIMA needs parsimony** — with n=29, any ARIMA with p+q > 3 overfits. The best ARIMA models in energy forecasting on short series are typically ARIMA(1,1,0), ARIMA(0,1,1), or ARIMA(1,1,1) (Hyndman & Khandakar, 2008).

3. **ML needs careful regularisation** — Random Forest and GB with default settings massively overfit on n=29. Conservative `max_depth=2–4` and `min_samples_leaf=3` are essential.

4. **RE penetration is genuinely hard to forecast** — the 2019 structural break means the training data (1990–2018, all hydro-era) has limited predictive power for the solar/wind transition era. This is not a modelling failure; it is a data limitation that requires expert scenario ranges rather than point forecasts.

5. **Fossil generation is the most predictable target** — it has strong autocorrelation (gas-dominated, slowly growing) without a structural break in the training window. ARIMA performs well here regardless of specification.

### Recommendations for ILF Dashboard (Notebook 07)
- Show bootstrap confidence bands on all demand forecasts
- For RE penetration: show scenario range (Accelerated / Baseline / Delayed) rather than a single point forecast
- Label which model drives each panel
- Add a note that 2019–2023 test MAPE is the best available measure of forecast reliability

### Next Steps
- **Notebook 05:** Investment Signal Detection — threshold-based rules on model outputs
- **Notebook 06:** Spatial Analysis — oblast-level demand disaggregation
- **Notebook 07:** Plotly Dash Dashboard — interactive scenario explorer